# 06 · Treino estendido: mais dados, mais modelos e consistência

O notebook 05 treinou cada modelo **uma vez** e mostrou que nenhum superou o FinBERT-PT-BR. Aqui verificamos se essa conclusão se sustenta:

1. **Consistência:** cada modelo é treinado com 5 sementes diferentes. Reportamos média ± desvio-padrão do F1 macro e um intervalo de confiança de 95% por *bootstrap* no teste. Uma diferença só é real se não couber no ruído entre sementes.
2. **Mais dados:** além das bases em português do notebook 05, usamos três bases em inglês, rotuladas por humanos: **SEntFiN** (manchetes com sentimento por empresa citada, o mais parecido com o nosso problema), **Twitter Financial News** e **Financial PhraseBank**.
3. **Mais modelos**, do clássico ao Transformer:

| # | modelo | dados de treino | aula |
|---|---|---|---|
| 1 | TF-IDF + Regressão Logística | português | 4 |
| 2 | BiLSTM e GRU (Keras) | português | 10 |
| 3 | embeddings de frase multilíngues (congelados) + Regressão Logística | português + inglês | 6 e 13 |
| 4 | **BERTimbau** (`neuralmind/bert-base-portuguese-cased`) com fine-tuning | português | 11 |
| 5 | **FinBERT-PT-BR** com fine-tuning adicional | português | 11 |
| 6 | **XLM-RoBERTa** (`FacebookAI/xlm-roberta-base`) com fine-tuning | português + inglês | 11 |
| 7 | XLM-RoBERTa treinado **só em inglês** e testado em português (transferência entre línguas) | inglês | 11 |
| — | FinBERT-PT-BR sem treino (linha de base do notebook 03) | — | 8 |

O teste continua sendo as **200 manchetes revisadas** (`data/teste_revisao.csv`), que nunca entram no treino nem na escolha de época.

**Como rodar no VS Code:**

1. Rode a célula **Ambiente** uma vez, com qualquer Python 3.10–3.12 como kernel. Ela cria o `.venv`, instala as bibliotecas (PyTorch com GPU, se houver placa NVIDIA) e registra o kernel **Python (portfolio-radar)**.
2. Troque o kernel para **Python (portfolio-radar)** (canto superior direito) e rode **todas as células a partir de "Configuração"**.
3. Escolha o `MODO` na célula de configuração: `"rapido"` (~10 min, para testar) ou `"completo"` (~4 a 6 h numa RTX 3060).

Cada treino salva o resultado em `RESULTADOS/rodadas/` assim que termina. Se o computador desligar, basta rodar de novo: as rodadas prontas são puladas. Deixe o Windows sem suspender durante o treino.

## Ambiente

Cria o ambiente virtual e instala tudo. Pode rodar de novo sem problema: o que já estiver instalado é pulado.

In [ ]:
import os
import shutil
import subprocess
import sys
import venv
from pathlib import Path

RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
VENV = RAIZ / ".venv"
PY = VENV / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
TEM_GPU_NVIDIA = shutil.which("nvidia-smi") is not None


def roda(*args):
    print(">", " ".join(str(a) for a in args))
    subprocess.run([str(a) for a in args], check=True)


def no_ambiente(codigo):
    """True se o código roda sem erro no Python do ambiente."""
    return subprocess.run([str(PY), "-c", codigo], capture_output=True).returncode == 0


if not PY.exists():
    print(f"Criando o ambiente em {VENV}")
    venv.create(VENV, with_pip=True)
roda(PY, "-m", "pip", "install", "--quiet", "--upgrade", "pip")
if TEM_GPU_NVIDIA and not no_ambiente("import torch, sys; sys.exit(0 if torch.version.cuda else 1)"):
    # PyTorch com CUDA. O índice padrão do pip só tem a versão para CPU no Windows.
    roda(PY, "-m", "pip", "install", "--quiet", "torch==2.14.1", "--index-url", "https://download.pytorch.org/whl/cu130")
roda(PY, "-m", "pip", "install", "--quiet", "-r", RAIZ / "requirements.txt")
for modelo_spacy in ["pt_core_news_sm", "pt_core_news_lg"]:
    if not no_ambiente(f"import {modelo_spacy}"):
        roda(PY, "-m", "spacy", "download", modelo_spacy)
roda(PY, "-m", "ipykernel", "install", "--user", "--name", "portfolio-radar", "--display-name", "Python (portfolio-radar)")
print("PyTorch com GPU:", no_ambiente("import torch, sys; sys.exit(0 if torch.cuda.is_available() else 1)"))
print("\nPronto. Troque o kernel para 'Python (portfolio-radar)' e continue a partir de 'Configuração'.")

## Configuração

In [ ]:
MODO = "completo"  # "completo" para o treino de verdade; "rapido" (~10 min) só para testar o notebook

CONFIG = {
    "rapido": {"sementes": [42], "amostra_treino": 3000, "epocas_transformer": 1, "epocas_rnn": 3},
    "completo": {"sementes": [42, 7, 123, 2024, 31337], "amostra_treino": None, "epocas_transformer": 2,
                 "epocas_rnn": 10},
}[MODO]

MODELOS = [  # comente uma linha para pular um modelo
    "tfidf_lr",
    "bilstm",
    "gru",
    "embeddings_lr",
    "bertimbau",
    "finbert_ajustado",
    "xlmr_pt_en",
    "xlmr_so_en",
]

In [ ]:
import json
import random
import time
import urllib.request
import warnings

os.environ["KERAS_BACKEND"] = "torch"  # Keras 3 roda sobre o PyTorch e usa a GPU
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(RAIZ / "src"))
from preprocessamento import limpa

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Modo: {MODO} · dispositivo: {DISPOSITIVO}" + (f" ({torch.cuda.get_device_name(0)})" if DISPOSITIVO == "cuda" else ""))
if DISPOSITIVO == "cpu" and MODO == "completo":
    print("AVISO: sem GPU, os Transformers vão levar muitas horas. Considere comentar bertimbau/finbert/xlmr.")

DADOS = RAIZ / "data"
RAW = DADOS / "raw"
RAW.mkdir(parents=True, exist_ok=True)
RESULTADOS = RAIZ / "Entregas" / "09-11-2026" / "treino_estendido" / MODO
RODADAS = RESULTADOS / "rodadas"
RODADAS.mkdir(parents=True, exist_ok=True)

ROTULOS = ["boa", "neutra", "ruim"]
INDICE = {r: i for i, r in enumerate(ROTULOS)}
SEMENTE_DADOS = 42  # a separação treino/validação é a mesma para todas as sementes de modelo
pd.set_option("display.max_colwidth", 110)


def fixa_semente(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

## Dados

Todas as bases são baixadas do Hugging Face para `data/raw/` (fora do git) na primeira execução.

| base | idioma | textos | rótulo original → nosso | licença |
|---|---|---|---|---|
| `lucas-leme/Sentiments-FinBERT-PT-BR` | pt | 503 | Positivo/Neutro/Negativo | Apache 2.0 |
| `heitorrosa/financial-sentiment-pt` | pt (traduzido) | ~46 mil | positive/neutral/negative | GPL-3.0 |
| `tasksource/sentfin` (SEntFiN) | en | ~7,5 mil manchetes | sentimento por entidade; usamos só manchetes em que todas as entidades têm o mesmo rótulo | MIT |
| `zeroshot/twitter-financial-news-sentiment` | en | ~12 mil | bearish → ruim, bullish → boa, neutral → neutra | MIT |
| `ChanceFocus/en-fpb` (Financial PhraseBank) | en | ~4,8 mil | positive/neutral/negative, rotulado pelo efeito no preço da ação | CC BY-NC-SA 3.0 (uso acadêmico) |

In [ ]:
HF = "https://huggingface.co/datasets"


def baixa(caminho_hf, nome):
    arq = RAW / nome
    if not arq.exists():
        print(f"Baixando {caminho_hf}")
        urllib.request.urlretrieve(f"{HF}/{caminho_hf}", arq)
    return arq


def le(caminho_hf, nome):
    arq = baixa(caminho_hf, nome)
    if nome.endswith(".parquet"):
        return pd.read_parquet(arq)
    if nome.endswith(".jsonl"):
        return pd.read_json(arq, lines=True)
    return pd.read_csv(arq)


def base(textos, rotulos, nome, idioma):
    return pd.DataFrame({"texto": list(textos), "rotulo": list(rotulos), "base": nome, "idioma": idioma})


# Português
lucas = le("lucas-leme/Sentiments-FinBERT-PT-BR/resolve/main/sentiments.csv", "finbert_ptbr_sentiments.csv")
lucas = lucas[lucas["sentiment"].isin(["Positivo", "Neutro", "Negativo"])]
pt_humano = base(lucas["text"], lucas["sentiment"].map({"Positivo": "boa", "Neutro": "neutra", "Negativo": "ruim"}),
                 "FinBERT-PT-BR (humano)", "pt")

heitor = le("heitorrosa/financial-sentiment-pt/resolve/main/final.jsonl", "financial_sentiment_pt.jsonl")
heitor = heitor[heitor["quality"] >= 0.6]
pt_traduzido = base(heitor["translated"], heitor["sentiment"].map({"positive": "boa", "neutral": "neutra", "negative": "ruim"}),
                    "financial-sentiment-pt (traduzido)", "pt")

# Inglês
EN_PARA_NOSSO = {"positive": "boa", "neutral": "neutra", "negative": "ruim"}
sentfin = pd.concat([le(f"tasksource/sentfin/resolve/main/data/{s}-00000-of-00001.parquet", f"sentfin_{s}.parquet")
                     for s in ["train", "validation", "test"]])
consistentes = sentfin.groupby("sentence1")["labels"].agg(lambda s: s.iloc[0] if s.nunique() == 1 else None).dropna()
en_sentfin = base(consistentes.index, consistentes.map(EN_PARA_NOSSO), "SEntFiN", "en")

twitter = pd.concat([le(f"zeroshot/twitter-financial-news-sentiment/resolve/main/sent_{s}.csv", f"twitter_fin_{s}.csv")
                     for s in ["train", "valid"]])
twitter["text"] = twitter["text"].str.replace(r"https?://\S+", "", regex=True).str.strip()
en_twitter = base(twitter["text"], twitter["label"].map({0: "ruim", 1: "boa", 2: "neutra"}), "Twitter Financial News", "en")

fpb_arqs = {"train": "train-00000-of-00001-ab9a3b4799b09589", "valid": "valid-00000-of-00001-303e4ba2afe838d4",
            "test": "test-00000-of-00001-8bd1e21c671fb670"}
fpb = pd.concat([le(f"ChanceFocus/en-fpb/resolve/main/data/{a}.parquet", f"fpb_{s}.parquet") for s, a in fpb_arqs.items()])
en_fpb = base(fpb["text"], fpb["answer"].map(EN_PARA_NOSSO), "Financial PhraseBank", "en")

# Teste
teste = pd.read_csv(DADOS / "teste_revisao.csv")
teste = teste[teste["rotulo"].isin(ROTULOS)].reset_index(drop=True)
y_teste = teste["rotulo"].map(INDICE).to_numpy()

todas = pd.concat([pt_humano, pt_traduzido, en_sentfin, en_twitter, en_fpb], ignore_index=True).dropna()
todas["texto"] = todas["texto"].astype(str).str.strip()
todas["limpo"] = todas["texto"].apply(limpa)
todas = todas[todas["limpo"].str.len() > 0].drop_duplicates("texto")
todas = todas[~todas["limpo"].isin(set(teste["manchete"].apply(limpa)))].reset_index(drop=True)  # teste fica inédito
pd.crosstab([todas["idioma"], todas["base"]], todas["rotulo"], margins=True)

### Conjuntos de treino

Cada combinação de idiomas tem a sua separação treino/validação (90/10, estratificada e sempre com a mesma semente). No modo rápido, o treino é uma amostra.

In [ ]:
def separa(idiomas):
    dados = todas[todas["idioma"].isin(idiomas)]
    treino, valid = train_test_split(dados, test_size=0.1, stratify=dados["rotulo"], random_state=SEMENTE_DADOS)
    if CONFIG["amostra_treino"]:
        treino = treino.sample(min(CONFIG["amostra_treino"], len(treino)), random_state=SEMENTE_DADOS)
        valid = valid.sample(min(CONFIG["amostra_treino"] // 5, len(valid)), random_state=SEMENTE_DADOS)
    return treino.reset_index(drop=True), valid.reset_index(drop=True)


CONJUNTOS = {"pt": separa(["pt"]), "pt_en": separa(["pt", "en"]), "en": separa(["en"])}
pd.DataFrame({nome: {"treino": len(t), "validação": len(v)} for nome, (t, v) in CONJUNTOS.items()}).T

## Rodadas

Cada função de modelo recebe a semente e o conjunto de dados e devolve as previsões no teste e o F1 macro na validação. A função `rodada` guarda tudo em `rodadas/<modelo>__s<semente>.json` e pula o que já existe.

In [ ]:
def metricas(y, previsto):
    return {"acuracia": accuracy_score(y, previsto), "f1_macro": f1_score(y, previsto, average="macro"),
            **{f"f1_{r}": f for r, f in zip(ROTULOS, f1_score(y, previsto, average=None, labels=range(3)))}}


def rodada(modelo, semente, funcao, conjunto):
    arq = RODADAS / f"{modelo}__s{semente}.json"
    if arq.exists():
        return json.loads(arq.read_text(encoding="utf-8"))
    print(f"\n=== {modelo} · semente {semente} · dados {conjunto} ===")
    fixa_semente(semente)
    inicio = time.time()
    previsto, f1_valid = funcao(*CONJUNTOS[conjunto], semente)
    r = {"modelo": modelo, "semente": semente, "dados": conjunto, "f1_valid": f1_valid,
         "minutos": round((time.time() - inicio) / 60, 2), **metricas(y_teste, previsto),
         "previsto": [ROTULOS[i] for i in previsto]}
    arq.write_text(json.dumps(r, ensure_ascii=False), encoding="utf-8")
    print(f"F1 macro: validação {f1_valid:.3f} · teste {r['f1_macro']:.3f} · {r['minutos']} min")
    if DISPOSITIVO == "cuda":
        torch.cuda.empty_cache()
    return r

### 1 · TF-IDF + Regressão Logística (aula 4)

In [ ]:
def tfidf_lr(treino, valid, semente):
    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    lr = LogisticRegression(max_iter=3000, class_weight="balanced", C=2.0, random_state=semente)
    lr.fit(tfidf.fit_transform(treino["limpo"]), treino["rotulo"].map(INDICE))
    f1_valid = f1_score(valid["rotulo"].map(INDICE), lr.predict(tfidf.transform(valid["limpo"])), average="macro")
    return lr.predict(tfidf.transform(teste["manchete"].apply(limpa))), f1_valid

### 2 · BiLSTM e GRU (aula 10)

A mesma arquitetura do notebook 05, agora com o Keras sobre o PyTorch para usar a GPU. O vocabulário é montado à mão (as 20 mil palavras mais frequentes do treino), no lugar do `TextVectorization`, que depende do TensorFlow.

In [ ]:
from collections import Counter

import keras
from keras import layers

MAX_TOKENS, MAX_LEN = 20_000, 40


def vocabulario(textos):
    contagem = Counter(p for t in textos for p in t.split())
    return {p: i + 2 for i, (p, _) in enumerate(contagem.most_common(MAX_TOKENS - 2))}  # 0 = padding, 1 = desconhecida


def para_indices(textos, vocab):
    X = np.zeros((len(textos), MAX_LEN), dtype="int64")
    for i, t in enumerate(textos):
        ids = [vocab.get(p, 1) for p in t.split()][:MAX_LEN]
        X[i, :len(ids)] = ids
    return X


def rede_recorrente(tipo):
    def treina(treino, valid, semente):
        keras.utils.set_random_seed(semente)
        vocab = vocabulario(treino["limpo"])
        X_tr, X_va, X_te = (para_indices(d, vocab) for d in (treino["limpo"], valid["limpo"], teste["manchete"].apply(limpa)))
        y_tr, y_va = treino["rotulo"].map(INDICE).to_numpy(), valid["rotulo"].map(INDICE).to_numpy()
        pesos = len(y_tr) / (3 * np.bincount(y_tr, minlength=3))

        recorrente = layers.Bidirectional(layers.LSTM(64)) if tipo == "bilstm" else layers.GRU(64)
        modelo = keras.Sequential([keras.Input(shape=(MAX_LEN,), dtype="int64"),
                                   layers.Embedding(MAX_TOKENS, 128, mask_zero=True), recorrente,
                                   layers.Dropout(0.3), layers.Dense(3, activation="softmax")])
        modelo.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy")

        melhor, melhores_pesos, sem_melhora = -1, None, 0
        for epoca in range(CONFIG["epocas_rnn"]):
            modelo.fit(X_tr, y_tr, batch_size=128, epochs=1, class_weight=dict(enumerate(pesos)), verbose=0)
            f1 = f1_score(y_va, modelo.predict(X_va, verbose=0).argmax(1), average="macro")
            print(f"  época {epoca + 1}: F1 validação {f1:.3f}")
            if f1 > melhor:
                melhor, melhores_pesos, sem_melhora = f1, modelo.get_weights(), 0
            else:
                sem_melhora += 1
                if sem_melhora == 2:  # parada antecipada
                    break
        modelo.set_weights(melhores_pesos)
        return modelo.predict(X_te, verbose=0).argmax(1), melhor
    return treina

### 3 · Embeddings de frase multilíngues + Regressão Logística

Um Transformer já treinado para gerar um vetor por frase (`paraphrase-multilingual-mpnet-base-v2`, da biblioteca sentence-transformers, aula 13) transforma cada texto em 768 números. Frases com o mesmo sentido ficam próximas mesmo em línguas diferentes, então dá para treinar com inglês e português juntos. O Transformer fica congelado: só a Regressão Logística é treinada.

In [ ]:
def embeddings_lr(treino, valid, semente):
    from sentence_transformers import SentenceTransformer
    codificador = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2", device=DISPOSITIVO)
    E_tr, E_va, E_te = (codificador.encode(list(t), batch_size=128, show_progress_bar=False, normalize_embeddings=True)
                        for t in (treino["texto"], valid["texto"], teste["manchete"]))
    lr = LogisticRegression(max_iter=3000, class_weight="balanced", C=1.0, random_state=semente)
    lr.fit(E_tr, treino["rotulo"].map(INDICE))
    del codificador
    return lr.predict(E_te), f1_score(valid["rotulo"].map(INDICE), lr.predict(E_va), average="macro")

### 4 a 7 · Fine-tuning de Transformers (aula 11)

Um laço de treino simples em PyTorch, igual para todos os modelos:

- o texto vai **sem limpeza**: o tokenizador do modelo divide as palavras em subpalavras, e maiúsculas e números ajudam um modelo `cased`;
- otimizador AdamW, taxa de aprendizado 2e-5 com aquecimento linear, lotes de 32, precisão mista (fp16) na GPU;
- a perda é ponderada pelas classes, como nas outras redes;
- ao fim de cada época medimos o F1 macro na validação e ficamos com a melhor época.

O FinBERT-PT-BR já tem uma camada de saída para positivo/negativo/neutro. Mapeamos os nossos rótulos para os dele, assim o fine-tuning parte do que ele já sabe.

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
MAX_LEN_TRANSFORMER = 96


def transformer(nome_hf):
    def treina(treino, valid, semente):
        tokenizador = AutoTokenizer.from_pretrained(nome_hf)
        modelo = AutoModelForSequenceClassification.from_pretrained(nome_hf, num_labels=3)
        dele = {k.upper(): v for k, v in (modelo.config.label2id or {}).items()}
        if {"POSITIVE", "NEUTRAL", "NEGATIVE"} <= set(dele):  # FinBERT-PT-BR: reaproveita a camada de saída
            nosso_para_modelo = [dele["POSITIVE"], dele["NEUTRAL"], dele["NEGATIVE"]]
        else:
            nosso_para_modelo = [0, 1, 2]
        modelo.to(DISPOSITIVO)

        def lotes(textos, rotulos=None, embaralha=False):
            idx = list(range(len(textos)))
            gerador = torch.Generator().manual_seed(semente)
            for b in DataLoader(idx, batch_size=32, shuffle=embaralha, generator=gerador):
                enc = tokenizador([textos[i] for i in b], truncation=True, max_length=MAX_LEN_TRANSFORMER,
                                  padding=True, return_tensors="pt").to(DISPOSITIVO)
                y = None if rotulos is None else torch.tensor([nosso_para_modelo[rotulos[i]] for i in b], device=DISPOSITIVO)
                yield enc, y

        def preve(textos):
            modelo.eval()
            saidas = []
            with torch.no_grad(), torch.autocast(DISPOSITIVO, enabled=DISPOSITIVO == "cuda"):
                for enc, _ in lotes(textos):
                    saidas.append(modelo(**enc).logits.float().cpu())
            logits = torch.cat(saidas)[:, nosso_para_modelo]  # volta para a ordem boa/neutra/ruim
            return logits.argmax(1).numpy()

        textos_tr, y_tr = treino["texto"].tolist(), treino["rotulo"].map(INDICE).tolist()
        y_va = valid["rotulo"].map(INDICE).to_numpy()
        contagem = np.bincount(y_tr, minlength=3)
        peso_nosso = len(y_tr) / (3 * contagem)
        peso = torch.zeros(3)
        for i_nosso, i_modelo in enumerate(nosso_para_modelo):
            peso[i_modelo] = peso_nosso[i_nosso]
        perda = torch.nn.CrossEntropyLoss(weight=peso.to(DISPOSITIVO))

        otimizador = torch.optim.AdamW(modelo.parameters(), lr=2e-5, weight_decay=0.01)
        passos = CONFIG["epocas_transformer"] * ((len(textos_tr) + 31) // 32)
        agenda = get_linear_schedule_with_warmup(otimizador, int(0.06 * passos), passos)
        escala = torch.amp.GradScaler(enabled=DISPOSITIVO == "cuda")

        melhor, melhor_estado = -1, None
        for epoca in range(CONFIG["epocas_transformer"]):
            modelo.train()
            for passo, (enc, y) in enumerate(lotes(textos_tr, y_tr, embaralha=True)):
                with torch.autocast(DISPOSITIVO, enabled=DISPOSITIVO == "cuda"):
                    loss = perda(modelo(**enc).logits.float(), y)
                otimizador.zero_grad()
                escala.scale(loss).backward()
                escala.unscale_(otimizador)
                torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
                escala.step(otimizador)
                escala.update()
                agenda.step()
                if passo % 200 == 0:
                    print(f"  época {epoca + 1} · passo {passo}/{passos // CONFIG['epocas_transformer']} · perda {loss.item():.3f}")
            f1 = f1_score(y_va, preve(valid["texto"].tolist()), average="macro")
            print(f"  época {epoca + 1}: F1 validação {f1:.3f}")
            if f1 > melhor:
                melhor = f1
                melhor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
        modelo.load_state_dict(melhor_estado)
        previsto = preve(teste["manchete"].tolist())
        del modelo, otimizador
        return previsto, melhor
    return treina

## Treino

Esta é a célula demorada. Os modelos determinísticos (TF-IDF e embeddings congelados) rodam com uma semente só.

In [ ]:
PLANO = {
    "tfidf_lr": (tfidf_lr, "pt", False),
    "bilstm": (rede_recorrente("bilstm"), "pt", True),
    "gru": (rede_recorrente("gru"), "pt", True),
    "embeddings_lr": (embeddings_lr, "pt_en", False),
    "bertimbau": (transformer("neuralmind/bert-base-portuguese-cased"), "pt", True),
    "finbert_ajustado": (transformer("lucas-leme/FinBERT-PT-BR"), "pt", True),
    "xlmr_pt_en": (transformer("FacebookAI/xlm-roberta-base"), "pt_en", True),
    "xlmr_so_en": (transformer("FacebookAI/xlm-roberta-base"), "en", True),
}

inicio = time.time()
rodadas = []
for nome in MODELOS:
    funcao, conjunto, varia_semente = PLANO[nome]
    for semente in CONFIG["sementes"] if varia_semente else CONFIG["sementes"][:1]:
        rodadas.append(rodada(nome, semente, funcao, conjunto))
print(f"\nTreino concluído em {(time.time() - inicio) / 60:.0f} min")

## Linha de base: FinBERT-PT-BR sem treino

As previsões do notebook 03 (`Entregas/27-10-2026/resultado_sentimentos.csv`), nas mesmas 200 manchetes.

In [ ]:
finbert = pd.read_csv(RAIZ / "Entregas" / "27-10-2026" / "resultado_sentimentos.csv").set_index("id")
prev_finbert = teste["id"].map(finbert["previsto"]).map(INDICE).to_numpy()
rodadas.append({"modelo": "finbert_sem_treino", "semente": 0, "dados": "—", "f1_valid": np.nan, "minutos": 0,
                **metricas(y_teste, prev_finbert), "previsto": [ROTULOS[i] for i in prev_finbert]})

## Resultados

### Consistência entre sementes

Média ± desvio-padrão do F1 macro no teste. Desvio grande quer dizer que o resultado de uma rodada só (como no notebook 05) não é confiável.

In [ ]:
NOMES = {"tfidf_lr": "TF-IDF + LR", "bilstm": "BiLSTM", "gru": "GRU", "embeddings_lr": "Embeddings multilíngues + LR",
         "bertimbau": "BERTimbau (fine-tuning)", "finbert_ajustado": "FinBERT-PT-BR (fine-tuning)",
         "xlmr_pt_en": "XLM-R (pt + en)", "xlmr_so_en": "XLM-R (só en → pt)", "finbert_sem_treino": "FinBERT-PT-BR (sem treino)"}

res = pd.DataFrame(rodadas)
res["nome"] = res["modelo"].map(NOMES)
consistencia = res.groupby("nome").agg(
    dados=("dados", "first"), rodadas=("semente", "size"),
    f1_teste_media=("f1_macro", "mean"), f1_teste_desvio=("f1_macro", "std"),
    f1_teste_min=("f1_macro", "min"), f1_teste_max=("f1_macro", "max"),
    f1_validacao=("f1_valid", "mean"), acuracia=("acuracia", "mean"),
    f1_boa=("f1_boa", "mean"), f1_neutra=("f1_neutra", "mean"), f1_ruim=("f1_ruim", "mean"),
    minutos=("minutos", "sum"),
).sort_values("f1_teste_media", ascending=False).round(3)
consistencia

In [ ]:
ordem = consistencia.index[::-1]
fig, ax = plt.subplots(figsize=(9, 0.5 * len(ordem) + 1))
for i, nome in enumerate(ordem):
    valores = res.loc[res["nome"] == nome, "f1_macro"]
    ax.scatter(valores, [i] * len(valores), alpha=0.6, color="tab:blue")
    ax.plot(valores.mean(), i, "k|", markersize=18)
ax.axvline(res.loc[res["modelo"] == "finbert_sem_treino", "f1_macro"].iloc[0], color="tab:red", ls="--",
           label="FinBERT-PT-BR sem treino")
ax.set_yticks(range(len(ordem)), ordem)
ax.set_xlabel("F1 macro no teste (cada ponto é uma semente; o traço é a média)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### Intervalo de confiança e comparação com o FinBERT

O teste tem só 200 manchetes, então o F1 também varia com **quais** manchetes caíram no teste. O *bootstrap* sorteia 2.000 testes de 200 manchetes com reposição e recalcula o F1 em cada um:

- **IC 95%:** faixa onde ficam 95% desses F1;
- **P(melhor que FinBERT):** em que fração dos sorteios o modelo bate o FinBERT sem treino nas mesmas manchetes (*bootstrap* pareado). Acima de 95%, a vantagem é consistente.

Para os modelos com várias sementes, usamos o **voto da maioria** entre as sementes, que também mostra se juntar rodadas ajuda.

In [ ]:
rng = np.random.default_rng(0)
amostras = rng.integers(0, len(y_teste), size=(2000, len(y_teste)))


def f1_boot(previsto):
    return np.array([f1_score(y_teste[a], previsto[a], average="macro") for a in amostras])


boot_finbert = f1_boot(prev_finbert)
linhas = []
for nome, grupo in res.groupby("nome"):
    votos = np.array([[INDICE[r] for r in p] for p in grupo["previsto"]])
    maioria = np.array([np.bincount(c, minlength=3).argmax() for c in votos.T])
    boot = f1_boot(maioria)
    linhas.append({"modelo": nome, "F1 macro (voto)": f1_score(y_teste, maioria, average="macro"),
                   "IC 95% inferior": np.percentile(boot, 2.5), "IC 95% superior": np.percentile(boot, 97.5),
                   "P(melhor que FinBERT)": (boot > boot_finbert).mean()})
comparacao = pd.DataFrame(linhas).set_index("modelo").sort_values("F1 macro (voto)", ascending=False).round(3)
comparacao

### Onde o melhor modelo acerta e erra

In [ ]:
melhor = comparacao.index[0] if comparacao.index[0] != NOMES["finbert_sem_treino"] else comparacao.index[1]
grupo = res[res["nome"] == melhor]
votos = np.array([[INDICE[r] for r in p] for p in grupo["previsto"]])
teste["melhor"] = [ROTULOS[np.bincount(c, minlength=3).argmax()] for c in votos.T]
teste["finbert"] = [ROTULOS[i] for i in prev_finbert]
print(f"Melhor modelo: {melhor}\n")
print(pd.crosstab(teste["rotulo"], teste["melhor"], rownames=["revisado"], colnames=["previsto"]))

proventos = teste["manchete"].str.contains("JCP|dividend|provento|juros sobre capital", case=False) & (teste["rotulo"] == "boa")
print(f"\nManchetes de proventos rotuladas como boas: {proventos.sum()} · "
      f"{melhor} acerta {(teste.loc[proventos, 'melhor'] == 'boa').sum()} · FinBERT acerta {(teste.loc[proventos, 'finbert'] == 'boa').sum()}")
teste.loc[teste["melhor"] != teste["rotulo"], ["empresa", "manchete", "rotulo", "melhor", "finbert"]].head(20)

## Resultados salvos

In [ ]:
consistencia.to_csv(RESULTADOS / "consistencia.csv")
comparacao.to_csv(RESULTADOS / "comparacao_bootstrap.csv")
res.drop(columns="previsto").to_csv(RESULTADOS / "rodadas.csv", index=False)
teste[["id", "empresa", "manchete", "rotulo", "melhor", "finbert"]].to_csv(RESULTADOS / "previsoes_melhor_modelo.csv", index=False)
print(f"Salvo em {RESULTADOS}")

## Análise

*Preencher depois do treino completo*, respondendo:

- **Consistência:** o desvio entre sementes da BiLSTM e da GRU é grande o bastante para mudar a conclusão do notebook 05?
- **Dados:** o XLM-R treinado só em inglês funciona em português? Juntar inglês e português ajuda?
- **Modelos:** algum Transformer com fine-tuning supera o FinBERT sem treino com P > 95%? Qual o custo em minutos?
- **Erros:** as manchetes de proventos (JCP, dividendos) continuam erradas? Se sim, o problema está no critério das bases de treino, e só dados rotulados no nosso critério resolvem.